# D13 · Robot Ethernet - VLANs, spanning tree, PTP and EtherCAT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D13_robot_ethernet_vlans_spanning_tree_ptp_and_ether.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 55 min

**Skill Hub concepts:** Virtual local area network; Rapid spanning tree protocol; Precision Time Protocol; Two-way time transfer; EtherCAT processing on the fly

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D13")

## Map: Ethernet features a robot network leans on

**802.1Q VLAN tag** (inserted after the source MAC):

```
 dst MAC (6) | src MAC (6) | TPID 0x8100 (2) | TCI: PCP 3 bits | DEI 1 | VID 12 (2) | EtherType (2) | payload
```
PCP = priority (0..7) for queueing; VID separates traffic (cameras vs. control) on shared switches.

**Rapid spanning tree (802.1w):** switches elect the root bridge (lowest bridge ID = priority, then MAC), compute the
least-cost path to it, and block redundant ports (alternate/backup roles) so a ring cannot loop broadcasts forever;
when a link fails a blocked port takes over in well under a second.

**Two-way time transfer (PTP, IEEE 1588):**

```
 master  t1 ──Sync──────────────▶ t2  slave
 master  t4 ◀──────Delay_Req──── t3  slave
 offset = ((t2 - t1) - (t4 - t3)) / 2        delay = ((t2 - t1) + (t4 - t3)) / 2
```
Assumes the path is symmetric; an asymmetry $a$ (forward minus reverse delay) shows up as an offset error of $a/2$.
Hardware timestamps in the PHY and transparent clocks in switches remove most of the error.

**EtherCAT:** one standard Ethernet frame leaves the master and passes through every slave in a line; each slave reads
its outputs and inserts its inputs **while the frame streams through** (nanoseconds per node), the last slave sends it
back. Each datagram has a working counter (WKC) that addressed slaves increment: read +1, write +2 (so read-write
success = +3 per slave). The master compares the WKC with the expected value every cycle.

## Task 1: parse a VLAN tag

In [ ]:
def parse_vlan(frame: bytes):
    """Return (vid, pcp, dei, ethertype) for a tagged frame, or None if untagged."""
    raise NotImplementedError  # TODO

lab.check("vlan", parse_vlan)

In [ ]:
%%writefile d13_vlan.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d13.h
// Grade it with lab.check("vlan", "d13_vlan.c"). For C++, name the file .cpp in both places.
#include "d13.h"

bool parse_vlan(const uint8_t *frame, size_t len, uint16_t *vid, uint8_t *pcp, uint8_t *dei, uint16_t *ethertype) {
    (void)frame; (void)len;
    *vid = 0; *pcp = 0; *dei = 0; *ethertype = 0;
    return false; /* TODO: TPID 0x8100 at offset 12, then TCI = PCP(3) DEI(1) VID(12), then the EtherType */
}

In [ ]:
# lab.check("vlan", "d13_vlan.c")  # uncomment after writing the C/C++ cell above

## Task 2: PTP offset and delay, and the cost of asymmetry

In [ ]:
def ptp(t1, t2, t3, t4):
    """Return (offset, delay): offset = slave clock minus master clock."""
    raise NotImplementedError  # TODO

def simulate_exchange(true_offset, d_forward, d_reverse, t1=0.0, t3_after_t2=5.0):
    """Generate the four timestamps a real exchange would produce (slave stamps include true_offset)."""
    raise NotImplementedError  # TODO

lab.check("ptp", ptp, simulate_exchange)

In [ ]:
%%writefile d13_ptp.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d13.h
// Grade it with lab.check("ptp", "d13_ptp.c"). For C++, name the file .cpp in both places.
#include "d13.h"

void ptp(double t1, double t2, double t3, double t4, double *offset, double *delay) {
    (void)t1; (void)t2; (void)t3; (void)t4;
    *offset = 0; *delay = 0; /* TODO */
}

void simulate_exchange(double true_offset, double d_forward, double d_reverse, double t1, double t3_after_t2, double *t4_out) {
    (void)true_offset; (void)d_forward; (void)d_reverse; (void)t3_after_t2;
    t4_out[0] = t1; t4_out[1] = t4_out[2] = t4_out[3] = 0; /* TODO: fill t1, t2, t3, t4 */
}

In [ ]:
# lab.check("ptp", "d13_ptp.c")  # uncomment after writing the C/C++ cell above

## Task 3: root bridge and EtherCAT working counter

In [ ]:
def root_bridge(bridges):
    """bridges: list of (priority, mac_string). Return the root's mac."""
    raise NotImplementedError  # TODO

def expected_wkc_lrw(slaves):
    """slaves: list of (has_inputs, has_outputs) mapped into one logical read-write datagram."""
    raise NotImplementedError  # TODO

lab.check("rstp_ecat", root_bridge, expected_wkc_lrw)

In [ ]:
%%writefile d13_rstp_ecat.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d13.h
// Grade it with lab.check("rstp_ecat", "d13_rstp_ecat.c"). For C++, name the file .cpp in both places.
#include "d13.h"

size_t root_bridge(const uint16_t *priority, const char **mac, size_t n) {
    (void)priority; (void)mac; (void)n;
    return 0; /* TODO: lowest priority, then lowest MAC */
}

int expected_wkc_lrw(const bool *has_inputs, const bool *has_outputs, size_t n) {
    (void)has_inputs; (void)has_outputs; (void)n;
    return 0; /* TODO: +1 per read, +2 per write */
}

In [ ]:
# lab.check("rstp_ecat", "d13_rstp_ecat.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** TCI value (integer) for VLAN 100 with priority 5, DEI 0?

**M2 (Apply).** t1 = 1000, t2 = 1060, t3 = 1100, t4 = 1120 (microseconds). Slave offset?

**M3 (Apply).** Expected working counter for one logical read-write datagram covering 4 servo drives that each have
inputs and outputs?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Forward path delay 10 us, reverse 14 us, true offset 0. What does PTP report?
- a) offset 0, delay 12 us
- b) offset -2 us, delay 12 us: half the asymmetry appears as a fake offset
- c) offset 4 us

**K2 (Evaluate).** Explain why EtherCAT reaches sub-microsecond jitter with hundreds of I/O points while standard
switched Ethernet with UDP cannot, and what the working counter buys the master.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d13.h`).
3. In `roboprep/checks/d13.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()